# External verification of `JCVISYN3A_0876 × _0878` synth-lethal prediction

Runs four external-database lookups for the two proteins (AVX55011.1 and AVX55013.1, the gene products of JCVISYN3A_0876 and JCVISYN3A_0878):

1. **UniProt REST** — find UniProt entries cross-referenced from these RefSeq accessions.
2. **UniProt orthology search** — Mycoplasma genus + amino-acid permease keyword.
3. **InterPro / Pfam (InterProScan REST)** — submit each protein sequence for domain annotation.
4. **NCBI BLAST (qblast REST)** — submit each sequence against UniProtKB for homologs.

All four fail in the Claude Code sandbox (proxy blocks every relevant database); they all work from Colab. Results write to `outputs/synthlet/0876_x_0878_external_lookup.json` and auto-push back to the branch.

**Wall estimate:** ~5 min for layers 1+2, ~10-30 min for layer 3 (InterProScan polls), ~5-15 min for layer 4 (NCBI BLAST polls). Skip slow cells if you're impatient — the fast ones are the most important.

**What success looks like:** if any of the four sources independently confirms that 0876 and 0878 are paralogous amino-acid permeases (Pfam family PF00324 / IPR004841 / similar), the SBML's GPR-OR redundancy assignment gets external biological corroboration without wet-lab work.

In [ ]:
# Cell 1 — install deps + clone repo + PAT prompt.
!pip install -q requests>=2.31 biopython>=1.83

import os, subprocess, getpass
BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
REPO_URL = "https://github.com/Nikku03/cell.git"
REPO_DIR = "/content/cell"

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if r.returncode != 0: raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    _run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    _run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    _run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
%cd /content/cell

if not os.environ.get("GITHUB_PAT", "").strip():
    pat = getpass.getpass("Paste your GitHub PAT (input hidden): ").strip()
    if not pat: raise ValueError("empty PAT")
    os.environ["GITHUB_PAT"] = pat
print(f"PAT set ({len(os.environ['GITHUB_PAT'])} chars)")

In [ ]:
# Cell 2 — extract sequences from in-repo syn3A.gb (re-fetch from
# Luthey-Schulten upstream if missing).
import urllib.request
from pathlib import Path
from Bio import SeqIO

STAGED = Path("cell_sim/data/Minimal_Cell_ComplexFormation/input_data")
STAGED.mkdir(parents=True, exist_ok=True)
GB = STAGED / "syn3A.gb"
if not GB.exists():
    print("staging syn3A.gb from Luthey-Schulten upstream...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Luthey-Schulten-Lab/"
        "Minimal_Cell_ComplexFormation/master/input_data/syn3A.gb", GB,
    )

SEQS = {}
rec = next(SeqIO.parse(GB, "genbank"))
for f in rec.features:
    if f.type != "CDS": continue
    lt = (f.qualifiers.get("locus_tag") or [""])[0]
    if lt in {"JCVISYN3A_0876", "JCVISYN3A_0878"}:
        SEQS[lt] = {
            "protein_id": (f.qualifiers.get("protein_id") or [""])[0],
            "product": (f.qualifiers.get("product") or [""])[0],
            "sequence": (f.qualifiers.get("translation") or [""])[0],
        }
for lt, d in SEQS.items():
    print(f"{lt}  protein_id={d['protein_id']}  len={len(d['sequence'])}aa  product={d['product']!r}")

In [ ]:
# Cell 3 — Layer 1: UniProt REST search by RefSeq accession.
# Looks up the two proteins via cross-reference from RefSeq protein
# accession (AVX*) to UniProtKB. Fast (seconds).
import json, requests, time

def _uniprot_search(query, fields="accession,protein_name,organism_name,length,xref_tcdb,xref_pfam"):
    url = "https://rest.uniprot.org/uniprotkb/search"
    r = requests.get(url, params={
        "query": query, "format": "tsv", "fields": fields, "size": "5",
    }, headers={"User-Agent": "cell-sim-bot/1.0"}, timeout=30)
    r.raise_for_status()
    return r.text

L1 = {}
for lt, d in SEQS.items():
    pid = d["protein_id"].split(".")[0]   # AVX55011 (drop version)
    print(f"\n=== {lt} ({pid}) ===")
    try:
        body = _uniprot_search(f"xref:refseq-{pid}")
    except Exception as e:
        body = f"FAIL: {type(e).__name__}: {e}"
    print(body[:600])
    L1[lt] = body
    time.sleep(0.5)

In [ ]:
# Cell 4 — Layer 2: UniProt orthology search.
# Find amino-acid permeases in Mycoplasma genus that share product
# annotation. If 0876/0878 have well-characterised orthologs in any
# of these organisms, those orthologs may have published phenotype
# data we can cite.
MYCO_TAXA = [
    (243273, "M_genitalium_G37"),
    (272634, "M_pneumoniae_M129"),
    (272632, "M_mycoides_subsp_mycoides"),
    (2027849, "M_mycoides_JCVI_Syn3.0"),
]
L2 = {}
for tax, label in MYCO_TAXA:
    print(f"\n=== amino acid permeases in {label} (taxid {tax}) ===")
    try:
        body = _uniprot_search(
            f'(taxonomy_id:{tax}) AND (protein_name:"amino acid permease")',
        )
    except Exception as e:
        body = f"FAIL: {type(e).__name__}: {e}"
    L2[label] = body
    print(body[:500])
    time.sleep(0.5)

In [ ]:
# Cell 5 — Layer 3: InterProScan REST submission.
# Submits both sequences for domain annotation. SLOW (each polls every
# 30 s for up to ~10 min total).
#
# Skip this cell if you only want the fast results — the UniProt
# layers 1-2 already give Pfam/TCDB cross-refs when entries exist.
import requests, time

IPRSCAN_BASE = "https://www.ebi.ac.uk/Tools/services/rest/iprscan5"
EMAIL = "cell-sim-bot@noreply.local"

def submit_iprscan(seq):
    r = requests.post(
        f"{IPRSCAN_BASE}/run",
        data={
            "email": EMAIL, "sequence": seq,
            "appl": "Pfam,InterPro",
            "goterms": "false", "pathways": "false",
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.text.strip()

def poll_iprscan(job_id, timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        r = requests.get(f"{IPRSCAN_BASE}/status/{job_id}", timeout=15)
        s = r.text.strip()
        if s == "FINISHED": return
        if s in ("ERROR", "FAILURE", "NOT_FOUND"):
            raise RuntimeError(f"InterProScan job {job_id} status: {s}")
        time.sleep(20)
    raise TimeoutError(f"InterProScan {job_id} did not finish in {timeout}s")

def fetch_iprscan_tsv(job_id):
    r = requests.get(f"{IPRSCAN_BASE}/result/{job_id}/tsv", timeout=30)
    r.raise_for_status()
    return r.text

L3 = {}
for lt, d in SEQS.items():
    print(f"\n=== InterProScan: {lt} ({len(d['sequence'])} aa) ===")
    try:
        job = submit_iprscan(d["sequence"])
        print(f"  job: {job}; polling...")
        poll_iprscan(job)
        tsv = fetch_iprscan_tsv(job)
        L3[lt] = tsv
        for line in tsv.splitlines()[:15]:
            cells = line.split("\t")
            if len(cells) >= 6:
                print(f"  {cells[3]:14s} {cells[4]:18s} {cells[5][:80]}")
    except Exception as e:
        L3[lt] = f"FAIL: {type(e).__name__}: {e}"
        print(f"  FAIL: {e}")

In [ ]:
# Cell 6 — Layer 4: NCBI BLAST against UniProtKB/Swiss-Prot.
# SLOW (5-15 min per query). Submits sequences to NCBI qblast and
# polls for results. Skip if you only want fast results.
import re

QBLAST = "https://blast.ncbi.nlm.nih.gov/Blast.cgi"

def blast_submit(seq):
    r = requests.post(QBLAST, data={
        "CMD": "Put", "PROGRAM": "blastp",
        "DATABASE": "swissprot", "QUERY": seq,
        "HITLIST_SIZE": 10, "FORMAT_TYPE": "Text",
    }, timeout=60)
    r.raise_for_status()
    rid = re.search(r"RID = (\S+)", r.text)
    rtoe = re.search(r"RTOE = (\d+)", r.text)
    if not rid: raise RuntimeError("no RID")
    return rid.group(1), int(rtoe.group(1)) if rtoe else 30

def blast_poll(rid, max_wait=900):
    t0 = time.time()
    while time.time() - t0 < max_wait:
        r = requests.get(QBLAST, params={
            "CMD": "Get", "FORMAT_OBJECT": "SearchInfo", "RID": rid,
        }, timeout=30)
        if "Status=READY" in r.text:
            return
        if "Status=FAILED" in r.text or "Status=UNKNOWN" in r.text:
            raise RuntimeError(f"BLAST {rid} failed")
        time.sleep(30)
    raise TimeoutError(f"BLAST {rid} did not finish in {max_wait}s")

def blast_fetch(rid):
    r = requests.get(QBLAST, params={
        "CMD": "Get", "RID": rid, "FORMAT_TYPE": "Text",
    }, timeout=60)
    r.raise_for_status()
    return r.text

L4 = {}
for lt, d in SEQS.items():
    print(f"\n=== NCBI BLAST: {lt} ===")
    try:
        rid, rtoe = blast_submit(d["sequence"])
        print(f"  rid={rid}  estimated wait {rtoe}s")
        time.sleep(min(rtoe, 60))
        blast_poll(rid)
        text = blast_fetch(rid)
        L4[lt] = text
        # Print top hits section
        in_hits = False
        for line in text.splitlines():
            if "Sequences producing significant alignments" in line:
                in_hits = True; print(line); continue
            if in_hits and line.strip() == "":
                break
            if in_hits:
                print(line[:140])
    except Exception as e:
        L4[lt] = f"FAIL: {type(e).__name__}: {e}"
        print(f"  FAIL: {e}")

In [ ]:
# Cell 7 — consolidate + auto-push.
import json
from pathlib import Path

out_dir = Path("outputs/synthlet")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "0876_x_0878_external_lookup.json"

results = {
    "queries": [
        {"locus": lt, "protein_id": d["protein_id"], "length": len(d["sequence"])}
        for lt, d in SEQS.items()
    ],
    "layer_1_uniprot_by_refseq": L1,
    "layer_2_uniprot_orthology": L2,
    "layer_3_interproscan": L3,
    "layer_4_ncbi_blast": L4,
}
out_path.write_text(json.dumps(results, indent=2, default=str))
print(f"wrote {out_path}  ({out_path.stat().st_size} bytes)")

# Auto-push.
pat = os.environ.get("GITHUB_PAT", "").strip()
if not pat:
    raise SystemExit("GITHUB_PAT not set; re-run Cell 1.")

_run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
_run(["git", "config", "user.name", "cell-sim-bot"])
_run(["git", "add", "-f", str(out_path)])
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
if not status.stdout.strip():
    print("nothing changed")
else:
    _run(["git", "commit", "-m",
          "Session 26: external biology-database lookup for 0876 x 0878"])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    _run(["git", "push", remote, BRANCH])
    print("\npush complete.")